# SCOUT demo

SCOUT searches for real failures with a cheap proxy. A mutual-information shortlist proposes regions the current labels do not explain. A control-variate estimator then re-ranks that shortlist using the proxy, so each real label is spent where the proxy says the failure score is high and the correction is trustworthy.

This notebook has two parts:

1. The same steps on a 2-D toy, where the real failure regions are two diamonds and the proxy is a shifted copy of that field.
2. The method, two ablations, and two baselines on a bundled KITTI ↔ Virtual KITTI pool (1,049 paired frames). No images are downloaded.


## Setup

In [ ]:
import os, subprocess, sys
from pathlib import Path

def find_root():
    start = Path.cwd().resolve()
    for cand in [start, *start.parents]:
        if (cand / "scout" / "mi.py").exists() and (cand / "scripts" / "toy2D" / "stepwise_viz.py").exists():
            return cand
    return None

root = find_root()
IN_COLAB = "google.colab" in sys.modules
if root is None and IN_COLAB:
    url = os.environ.get("SCOUT_REPO_URL", "REPO_URL_PLACEHOLDER")
    if "REPO_URL_PLACEHOLDER" in url:
        raise RuntimeError(
            "This checkout is not on the path. Set SCOUT_REPO_URL to the clone URL, "
            "or open the notebook from the repository root."
        )
    subprocess.check_call(["git", "clone", url, "/content/scout-code"])
    root = Path("/content/scout-code")

os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
print("repo:", root)


In [ ]:
import importlib.util, subprocess, sys
needed = ("torch", "gpytorch", "botorch", "sklearn", "pandas", "matplotlib", "joblib")
missing = [name for name in needed if importlib.util.find_spec(name) is None]
if missing:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "numpy", "scipy", "scikit-learn", "matplotlib", "torch",
        "gpytorch", "botorch", "pandas", "joblib", "pillow",
    ])
print("dependencies ready")


## Part 1 — 2-D toy, one step at a time

The real field is high inside two diamonds. The proxy uses the same shape with a small shift, so the two fields agree in some places and disagree in others. White dots are real labels. Gold stars are candidates.


In [ ]:
from scripts.toy2D.stepwise_viz import build_figures

captions = [
    "Step 1. Real target (left) and proxy (right). Outlines are the real diamonds.",
    "Step 2. A handful of noisy real labels. Everything else is still unlabeled.",
    "Step 3. GP posterior on the real labels (mean and uncertainty). Stars are the mutual-information shortlist among random designs.",
    "Step 4. Control variates re-score only the shortlist. The next batch (large stars) is where the proxy-corrected mean is highest.",
]
for caption, fig in zip(captions, build_figures(seed=7, n_init=16, grid_n=48)):
    print(caption)
    display(fig)


## Part 2 — KITTI ↔ Virtual KITTI

Each row of `data/kitti_vkitti/paired_detection_task_linked.csv` is one paired frame. `target_failure` is the real missed-car score. `proxy_failure` is the same score on the Virtual KITTI clone. Features are the bundled `X_rich.npy` matrix.

The cell below runs five methods from one shared initial set:

| Method | What it changes |
| --- | --- |
| SCOUT | MI shortlist, then local control variates that use the proxy |
| β = 0 | same loop with the proxy term removed |
| MI only | shortlist by mutual information, no surrogate |
| Random | uniform draws |
| Importance sampling | draws with probability increasing in the proxy failure |

The budget is short on purpose (8 initial labels, then 3 batches of 4). Full multi-seed sweeps are `scripts/kitti_vkitti/ablate_local_cv.py` and `compare_beta_ablation.py`. Higher cumulative mean target failure means real failures were found earlier.


In [ ]:
from scripts.kitti_vkitti.run_demo import main

fig, summary = main([
    "--n-init", "8",
    "--n-acq-iters", "3",
    "--batch-size", "4",
    "--bnn-epochs", "40",
    "--mi-shortlist", "40",
    "--cv-n-jobs", "2",
    "--output-dir", "outputs/kitti_vkitti/demo",
])
display(fig)
summary


## Reading the KITTI plot

The dotted line is the end of the shared initial set. Curves that rise faster after that line are spending real labels on higher-failure frames.

- **β = 0** keeps the MI shortlist and the target BNN, and drops the proxy correction. A gap above this curve is the value of the control variate.
- **MI only** never fits a failure surrogate. It tests the exploration term on its own.
- **Random** and **importance sampling** are the baselines. Importance sampling trusts the proxy directly, without the MI gate or the variance correction.

For the published hyper-parameters (15–20 acquisition rounds, 800 BNN epochs, three seeds), use the ablation drivers in `scripts/kitti_vkitti/`.
